In [54]:
import os
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)


# Load path and dataset

In [55]:
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"

paths = {
    "lms": DATA_DIR / "lms_loan_daily.csv",
    "clickstream": DATA_DIR / "feature_clickstream.csv",
    "attributes": DATA_DIR / "features_attributes.csv",
    "financials": DATA_DIR / "features_financials.csv",
}

for name, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

lms = pd.read_csv(paths["lms"])
click = pd.read_csv(paths["clickstream"])
attr = pd.read_csv(paths["attributes"])
fin = pd.read_csv(paths["financials"])

print({k: len(v) for k, v in [("lms", lms), ("clickstream", click), ("attributes", attr), ("financials", fin)]})

{'lms': 137500, 'clickstream': 215376, 'attributes': 12500, 'financials': 12500}


# Inspect `lms_loan_daily.csv`


In [56]:
for c in ["loan_start_date", "snapshot_date"]:
    lms[c] = pd.to_datetime(lms[c], errors="coerce")

n_loans = lms["loan_id"].nunique()
n_cust = lms["Customer_ID"].nunique()
rows_per_loan = lms.groupby("loan_id").size()

print("rows", len(lms))
print()
print("rows per loan_id")
print(rows_per_loan.describe())

rows 137500

rows per loan_id
count    12500.0
mean        11.0
std          0.0
min         11.0
25%         11.0
50%         11.0
75%         11.0
max         11.0
dtype: float64


In [57]:
print("loan_id nunique", n_loans)
print("Customer_ID nunique", n_cust)
print("rows", len(lms))
print("n_loans * 11:", n_loans * 11)
inst = sorted(lms["installment_num"].dropna().unique().tolist())
print("installment_num nunique", len(inst), inst)


loan_id nunique 12500
Customer_ID nunique 12500
rows 137500
n_loans * 11: 137500
installment_num nunique 11 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [58]:
for col in ["tenure", "loan_amt"]:
    nu = lms[col].nunique()
    print(col, "nunique", nu, lms[col].dropna().unique()[:8])

tenure nunique 1 [10]
loan_amt nunique 1 [10000]


### tenure and loan_amt are two constant values across all table

In [59]:
print("loan_start_date total distinct month values (M):", lms["loan_start_date"].dt.to_period("M").nunique())
print("earliest and latest loan_start_date ", lms["loan_start_date"].min(), lms["loan_start_date"].max())
print("earliest and latest snapshot_date ", lms["snapshot_date"].min(), lms["snapshot_date"].max())


loan_start_date total distinct month values (M): 25
earliest and latest loan_start_date  2023-01-01 00:00:00 2025-01-01 00:00:00
earliest and latest snapshot_date  2023-01-01 00:00:00 2025-11-01 00:00:00


In [60]:
for col in ["due_amt", "paid_amt", "overdue_amt", "balance"]:
    print(col)
    print(lms[col].describe(percentiles=[0.5, 0.9, 0.99]))

pos = (lms["overdue_amt"] > 0).sum()
pct = pos / len(lms)
print("overdue_amt > 0", int(pos), f"{pct:.3%}")

due_amt
count    137500.000000
mean        909.090909
std         287.480833
min           0.000000
50%        1000.000000
90%        1000.000000
99%        1000.000000
max        1000.000000
Name: due_amt, dtype: float64
paid_amt
count    137500.000000
mean        711.810909
std         485.726859
min           0.000000
50%        1000.000000
90%        1000.000000
99%        2000.000000
max        4000.000000
Name: paid_amt, dtype: float64
overdue_amt
count    137500.000000
mean        871.905455
std        2002.672544
min           0.000000
50%           0.000000
90%        4000.000000
99%        8000.000000
max       10000.000000
Name: overdue_amt, dtype: float64
balance
count    137500.000000
mean       5871.905455
std        3070.777575
min           0.000000
50%        7000.000000
90%       10000.000000
99%       10000.000000
max       10000.000000
Name: balance, dtype: float64
overdue_amt > 0 28876 21.001%


In [61]:
print("due_amt by installment_num")
print(lms.groupby("installment_num")["due_amt"].agg(["mean", "min", "max"]))

print(lms.isna().sum().sort_values(ascending=False).head(10))
display(lms.head(5))


due_amt by installment_num
                   mean     min     max
installment_num                        
0                   0.0     0.0     0.0
1                1000.0  1000.0  1000.0
2                1000.0  1000.0  1000.0
3                1000.0  1000.0  1000.0
4                1000.0  1000.0  1000.0
5                1000.0  1000.0  1000.0
6                1000.0  1000.0  1000.0
7                1000.0  1000.0  1000.0
8                1000.0  1000.0  1000.0
9                1000.0  1000.0  1000.0
10               1000.0  1000.0  1000.0
loan_id            0
Customer_ID        0
loan_start_date    0
tenure             0
installment_num    0
loan_amt           0
due_amt            0
paid_amt           0
overdue_amt        0
balance            0
dtype: int64


,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000,0.0,0.0,0.0,10000.0,2023-05-01
1,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2023-06-01
2,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2023-07-01
3,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,3,10000,1000.0,0.0,1000.0,8000.0,2023-08-01
4,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,4,10000,1000.0,2000.0,0.0,6000.0,2023-09-01


# Inspect `feature_clickstream.csv`

In [62]:
# Merge loan_start_date to clickstream for each customer
click["snapshot_date"] = pd.to_datetime(click["snapshot_date"], errors="coerce")

loan_start = (
    lms.groupby("Customer_ID", as_index=False)["loan_start_date"]
    .agg(loan_start_date="min")
)
click_m = click.merge(loan_start, on="Customer_ID", how="left")

In [63]:
fe_cols = [c for c in click.columns if c.startswith("fe_")]
rpc = click.groupby("Customer_ID").size()

In [64]:
print("clickstream rows", len(click), "customers", click["Customer_ID"].nunique())
print("rows per Customer_ID (describe):\n", rpc.describe())

clickstream rows 215376 customers 8974
rows per Customer_ID (describe):
 count    8974.0
mean       24.0
std         0.0
min        24.0
25%        24.0
50%        24.0
75%        24.0
max        24.0
dtype: float64


In [65]:
mask_after = click_m["snapshot_date"] > click_m["loan_start_date"]
print("Percentage of clickstream rows AFTER loan_start_date:", round(mask_after.mean(), 4))
print("Customers with post-application clickstream:", click_m.loc[mask_after, "Customer_ID"].nunique())

Percentage of clickstream rows AFTER loan_start_date: 0.6058
Customers with post-application clickstream: 8974


In [66]:
allowed = click_m[click_m["snapshot_date"] <= click_m["loan_start_date"]].groupby("Customer_ID").size()
print("Non-leakage rows per customer (<= application):\n", allowed.describe())

Non-leakage rows per customer (<= application):
 count    8974.000000
mean        9.461556
std         5.233160
min         1.000000
25%         5.000000
50%         9.000000
75%        14.000000
max        18.000000
dtype: float64


In [67]:
in_click = set(click["Customer_ID"])
in_lms = set(lms["Customer_ID"])
missing_click = len(in_lms - in_click)
print("LMS customers with NO clickstream rows:", missing_click, f"({missing_click / len(in_lms):.1%})")

LMS customers with NO clickstream rows: 3526 (28.2%)


# Inspect `features_attributes.csv`

In [68]:
attr["snapshot_date"] = pd.to_datetime(attr["snapshot_date"], errors="coerce")
attr_m = attr.merge(loan_start, on="Customer_ID", how="left")

same_day = attr_m["snapshot_date"].notna() & attr_m["loan_start_date"].notna() & (
    attr_m["snapshot_date"].dt.normalize() == attr_m["loan_start_date"].dt.normalize()
)

In [69]:
print("Attributes rows", len(attr), "customers", attr["Customer_ID"].nunique())
print("Snapshot_date == loan_start_date:", same_day.mean())

Attributes rows 12500 customers 12500
Snapshot_date == loan_start_date: 1.0


In [70]:
age_raw = attr["Age"].astype(str)
trailing_us = age_raw.str.endswith("_", na=False)
print("Age values with trailing underscore:", int(trailing_us.sum()))

Age values with trailing underscore: 637


In [71]:
age_num = pd.to_numeric(age_raw.str.rstrip("_"), errors="coerce")
outside = int(((age_num < 18) | (age_num > 100)).sum())
print(
    "Age (numeric, after stripping trailing _)\n"
    f"min: {age_num.min()}\n"
    f"max: {age_num.max()}\n"
    f"outside 18–100: {outside}"
)

Age (numeric, after stripping trailing _)
min: -500
max: 8678
outside 18–100: 988


In [72]:
occ = attr["Occupation"].astype(str)
print("Occupation placeholder _______ :", int((occ == "_______").sum()))


ssn = attr["SSN"].astype(str)
ssn_ok = ssn.str.fullmatch(r"\d{3}-\d{2}-\d{4}", na=False)
print("SSN valid NNN-GG-SSSS:", int(ssn_ok.sum()), f"({ssn_ok.mean():.1%})")
print("SSN not in correct format:", int((~ssn_ok).sum()), f"({(~ssn_ok).mean():.1%})")
display(attr.head(10))

Occupation placeholder _______ : 880
SSN valid NNN-GG-SSSS: 11797 (94.4%)
SSN not in correct format: 703 (5.6%)


,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
0,CUS_0x1000,Alistair Barrf,18,913-74-1218,Lawyer,2023-05-01
1,CUS_0x1009,Arunah,26,063-67-6938,Mechanic,2025-01-01
2,CUS_0x100b,Shirboni,19,#F%$D@*&8,Media_Manager,2024-03-01
3,CUS_0x1011,Schneyerh,44,793-05-8223,Doctor,2023-11-01
4,CUS_0x1013,Cameront,44,930-49-9615,Mechanic,2023-12-01
5,CUS_0x1015,Holtono,27,810-97-7024,Journalist,2023-08-01
6,CUS_0x1018,Felsenthalq,15,731-19-8119,Accountant,2023-11-01
7,CUS_0x1026,Josephv,52,500-62-9044,Manager,2023-10-01
8,CUS_0x102d,Neil Chatterjeex,31,692-71-7552,Entrepreneur,2024-01-01
9,CUS_0x102e,Rhysn,26,#F%$D@*&8,Scientist,2024-04-01
